
chmod +x scripts/setup_virtualenv.sh

./scripts/setup_virtualenv.sh

chmod +x scripts/run_app.sh

./run_app.sh


In [1]:
input_tracklist = """
1. mall grab - ive always liked grime
  (0:00)
2. dj seinfeld - ruff hysteria
  (2:00)
3. edmondson - diamond life
  (4:00)
4. dj boring - winona
  (6:00)
5. mall grab - father
  (8:00)
6. mall grab - cant
  (9:30)
7. harrison bdp - almond
  (12:30)
8. subjoi - hyperfunk
  (15:00)
9. edmondson - flamingo tripper
  (15:40)
10. four tet - planet  (18:30)
11. harrison bdp - confusion of sound
  (20:00)
12. mall grab - roadworks 
 (23:30)
13. mall grab - orange county
  (24:50)
14. mall grab - guap 
 (27:00)
15. dj seinfeld - jerry

  (29:30)
16. folamour - the power and the blessing of unity

  (30:45)
17. edmondson - on and on

  (32:45)
18. mall grab - happiness 

 (34:00)
19. dj seinfeld - flyin through sunshine

  (35:40)
20. subjoi - i got this feelin

  (38:20)
21. edmondson - saturdays

  (39:20)
22. folamour - each day is a first day 

 (41:00)
23. harrison bdp - goodbye

  (42:30)
24. subjoi - belle

  (44:30)
25. dj seinfeld - u 

 (46:30)
26. harrison bdp - born to squander

  (48:30)
27. folamour - oyabun

  (50:00)
28. mall grab - caught slippin

  (52:00)
29. subjoi - whispers 

 (52:50)
30. dj seinfeld - forget u

  (54:00)
31. harrison bdp - 28 days 

 (55:20)
32. folamour - ya just need 2 believe in yourself

  (56:00)
33. harrison bdp - decompression 

 (58:30)
34. subjoi - the way i feel

  (1:00:00)
35. harrison bdp - its foggy outside 

 (1:02:00)
36. dj seinfeld - time spent away from u 

 (1:03:30)
37. subjoi - love shy

  (1:05:00)
38. harrison bdp - last flight to wherever  (1:06:20)
"""

In [2]:
# Import necessary libraries
import os
from src.ai_funcs import get_tracklist

import pandas as pd
pd.set_option('display.max_rows', None)
from datetime import datetime, timezone
import json

# Process the input tracklist
processed_tracklist = get_tracklist(input_tracklist)

# Convert processed tracklist to DataFrame
def create_tracklist_dataframe(processed_tracklist):
    df = pd.DataFrame(processed_tracklist['tracks'])
    df['artist_title'] = df['artist'] + ' - ' + df['title']
    df['load_ts'] = datetime.now(timezone.utc)
    return df

# Create and display the DataFrame
df = create_tracklist_dataframe(processed_tracklist)

df


,artist,title,artist_title,load_ts
0,mall grab,ive always liked grime,mall grab - ive always liked grime,2024-09-09 03:03:35.174797+00:00
1,dj seinfeld,ruff hysteria,dj seinfeld - ruff hysteria,2024-09-09 03:03:35.174797+00:00
2,edmondson,diamond life,edmondson - diamond life,2024-09-09 03:03:35.174797+00:00
3,dj boring,winona,dj boring - winona,2024-09-09 03:03:35.174797+00:00
4,mall grab,father,mall grab - father,2024-09-09 03:03:35.174797+00:00
5,mall grab,cant,mall grab - cant,2024-09-09 03:03:35.174797+00:00
6,harrison bdp,almond,harrison bdp - almond,2024-09-09 03:03:35.174797+00:00
7,subjoi,hyperfunk,subjoi - hyperfunk,2024-09-09 03:03:35.174797+00:00
8,edmondson,flamingo tripper,edmondson - flamingo tripper,2024-09-09 03:03:35.174797+00:00
9,four tet,planet,four tet - planet,2024-09-09 03:03:35.174797+00:00


In [3]:
from deltalake import write_deltalake


In [4]:
write_deltalake("artifacts/tracklist_master", df, mode="append")



In [6]:
pip install openai==1.44.0


  Using cached openai-1.44.0-py3-none-any.whl.metadata (22 kB)
Using cached openai-1.44.0-py3-none-any.whl (367 kB)
  Attempting uninstall: openai
    Found existing installation: openai 0.28.0
    Uninstalling openai-0.28.0:
      Successfully uninstalled openai-0.28.0

[notice] A new release of pip is available: 23.3.1 -> 24.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
# Import necessary libraries
import os
from src.ai_funcs import get_tracklist
import pandas as pd
import json
from datetime import datetime, timezone
from deltalake import write_deltalake
from openai import OpenAI

# Set up OpenAI client (you'll need to set your API key as an environment variable)
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Function to process tracklist using OpenAI API
def process_tracklist_with_ai(input_tracklist):
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": "You are a helpful assistant that organizes music tracklists."},
            {"role": "user", "content": f"Please organize this tracklist into a JSON format with 'artist', 'title', and 'timestamp' fields:\n\n{input_tracklist}"}
        ]
    )
    return json.loads(response.choices[0].message.content)

# Function to create DataFrame from processed tracklist
def create_tracklist_dataframe(processed_tracklist):
    df = pd.DataFrame(processed_tracklist['tracks'])
    df['artist_title'] = df['artist'] + ' - ' + df['title']
    df['load_ts'] = datetime.now(timezone.utc)
    return df

# Main function to process tracklist and store in Delta Lake
def process_and_store_tracklist(input_tracklist):
    # Process the input tracklist using AI
    processed_tracklist = process_tracklist_with_ai(input_tracklist)
    
    # Convert processed tracklist to DataFrame
    df = create_tracklist_dataframe(processed_tracklist)
    
    # Display the DataFrame
    print(df)
    
    # Append to Delta Lake
    write_deltalake("artifacts/tracklist_master", df, mode="append")
    print("Tracklist appended to Delta Lake successfully.")

# Example usage
input_tracklist = """
1. mall grab - ive always liked grime (0:00)
2. dj seinfeld - ruff hysteria (2:00)
3. edmondson - diamond life (4:00)
// ... (rest of the tracklist)
"""

process_and_store_tracklist(input_tracklist)

# Function to deduplicate tracks in Delta Lake
def deduplicate_delta_lake():
    # Read the entire Delta table
    df = pd.read_delta("artifacts/tracklist_master")
    
    # Sort by load_ts and drop duplicates based on artist and title
    df_deduplicated = df.sort_values('load_ts').drop_duplicates(subset=['artist', 'title'], keep='last')
    
    # Write the deduplicated DataFrame back to Delta Lake
    write_deltalake("artifacts/tracklist_master", df_deduplicated, mode="overwrite")
    print("Tracks deduplicated successfully.")

# Run deduplication (you can call this periodically or as needed)
deduplicate_delta_lake()

JSONDecodeError: Expecting value: line 1 column 1 (char 0)